## R-1 Access Control

In [0]:
use catalog claims;

create schema if not exists gov;

In [0]:
create table if not exists gov.admin_users(user_name string)

In [0]:
create table if not exists gov.user_state (user_name string, state string)

In [0]:
create or replace function gov.mask_last4 (v string)
returns string
return 
case 
when exists (
    select user_name
    from gov.admin_users
    where user_name = current_user()
) 
then v
else 
    concat("****", right(v,4))
end;

In [0]:
ALTER TABLE claims.raw.members
  ALTER COLUMN national_id SET MASK claims.gov.mask_last4;

ALTER TABLE claims.raw.members
  ALTER COLUMN phone SET MASK claims.gov.mask_last4;

In [0]:
select * from claims.raw.members limit 5

In [0]:
create or replace function gov.claims_state_filter (p_hospital_id string)
returns boolean
return exists (
  SELECT 1
  FROM claims.gov.admin_users
  WHERE user_name = current_user()
) OR
exists (
  SELECT 1
  FROM claims.gov.user_state us
  JOIN claims.raw.hospitals h
    ON h.hospital_id = p_hospital_id
  WHERE us.user_name = current_user()
    AND us.state = h.state
)

In [0]:
ALTER TABLE claims.gold.claims_enriched
  SET ROW FILTER claims.gov.claims_state_filter ON (hospital_id);

In [0]:
insert into claims.gov.user_state values (current_user(), "Karnataka")

In [0]:
select distinct hospital_id from claims.gold.claims_enriched

Proof tables

In [0]:
-- State users proof table 
insert into claims.gov.user_state values(current_user(), "Karnataka")

In [0]:
create table claims.gov.proof_member_state as 
select c.claim_id, c.hospital_id, h.state from claims.gold.claims_enriched c join claims.raw.hospitals h on c.hospital_id = h.hospital_id

In [0]:
-- Masked member proof table 
create table claims.gov.proof_member_masked as
select * from claims.raw.members

In [0]:
-- Admin member proof table 
insert into claims.gov.admin_users values (current_user())

In [0]:
create table claims.gov.proof_member_admin as
select * from claims.raw.members

## R2 Discovery and Lineage

In [0]:
-- PII tags on sensitive data 
ALTER TABLE claims.raw.members ALTER COLUMN name SET TAGS ('pii' = 'true');
ALTER TABLE claims.raw.members ALTER COLUMN dob SET TAGS ('pii' = 'true');
ALTER TABLE claims.raw.members ALTER COLUMN national_id SET TAGS ('pii' = 'true');
ALTER TABLE claims.raw.members ALTER COLUMN phone SET TAGS ('pii' = 'true');
ALTER TABLE claims.raw.members ALTER COLUMN email SET TAGS ('pii' = 'true');

In [0]:
-- Documentation : table + column comments for all claims.gold tables

-- claims.gold.claims_enriched 
COMMENT ON TABLE claims.gold.claims_enriched IS
  'Claims with the plan and network facts that applied on each claim''s own admission date. Facts only — no eligibility or payability verdicts.';

ALTER TABLE claims.gold.claims_enriched ALTER COLUMN claim_id COMMENT 'Unique identifier for the claim.';
ALTER TABLE claims.gold.claims_enriched ALTER COLUMN member_id COMMENT 'Identifier of the member (patient) who filed the claim.';
ALTER TABLE claims.gold.claims_enriched ALTER COLUMN policy_id COMMENT 'Identifier of the policy covering the claim.';
ALTER TABLE claims.gold.claims_enriched ALTER COLUMN hospital_id COMMENT 'Identifier of the hospital where the member was admitted; joins to claims.raw.hospitals.';
ALTER TABLE claims.gold.claims_enriched ALTER COLUMN admission_date COMMENT 'Date the member was admitted.';
ALTER TABLE claims.gold.claims_enriched ALTER COLUMN discharge_date COMMENT 'Date the member was discharged.';
ALTER TABLE claims.gold.claims_enriched ALTER COLUMN admission_month COMMENT 'First day of the admission month; grouping key for monthly reporting.';
ALTER TABLE claims.gold.claims_enriched ALTER COLUMN diagnosis_code COMMENT 'Medical diagnosis code recorded on the claim.';
ALTER TABLE claims.gold.claims_enriched ALTER COLUMN procedure_code COMMENT 'Medical procedure code recorded on the claim.';
ALTER TABLE claims.gold.claims_enriched ALTER COLUMN is_pre_existing COMMENT 'True if the claim relates to a pre-existing condition.';
ALTER TABLE claims.gold.claims_enriched ALTER COLUMN claim_type COMMENT 'Processing type of the claim: cashless or reimbursement.';
ALTER TABLE claims.gold.claims_enriched ALTER COLUMN billed_amount COMMENT 'Total amount billed by the hospital for the claim.';
ALTER TABLE claims.gold.claims_enriched ALTER COLUMN room_rent_per_day COMMENT 'Daily room rent charged on the claim.';
ALTER TABLE claims.gold.claims_enriched ALTER COLUMN room_days COMMENT 'Number of room days charged on the claim.';
ALTER TABLE claims.gold.claims_enriched ALTER COLUMN status COMMENT 'Lifecycle status of the claim: submitted, approved, rejected, or settled.';
ALTER TABLE claims.gold.claims_enriched ALTER COLUMN approved_amount COMMENT 'Amount approved by the insurer for settlement.';
ALTER TABLE claims.gold.claims_enriched ALTER COLUMN rejection_code COMMENT 'Reason code set when the claim was rejected; null otherwise.';
ALTER TABLE claims.gold.claims_enriched ALTER COLUMN submitted_ts COMMENT 'Timestamp when the claim was submitted.';
ALTER TABLE claims.gold.claims_enriched ALTER COLUMN settled_ts COMMENT 'Timestamp when the claim was settled; null while it is open.';
ALTER TABLE claims.gold.claims_enriched ALTER COLUMN channel COMMENT 'Submission channel: app, email, or hospital_portal.';
ALTER TABLE claims.gold.claims_enriched ALTER COLUMN plan_code_at_admission COMMENT 'Plan code in effect on admission_date, resolved from dim_policy_scd2.';
ALTER TABLE claims.gold.claims_enriched ALTER COLUMN plan_family COMMENT 'Family of the plan in effect: individual, group, or senior.';
ALTER TABLE claims.gold.claims_enriched ALTER COLUMN room_rent_cap_per_day COMMENT 'Per-day room rent cap defined by the plan on admission_date.';
ALTER TABLE claims.gold.claims_enriched ALTER COLUMN policy_start_date COMMENT 'Start date of the policy.';
ALTER TABLE claims.gold.claims_enriched ALTER COLUMN policy_tenure_months COMMENT 'Total policy tenure in months.';
ALTER TABLE claims.gold.claims_enriched ALTER COLUMN length_of_stay COMMENT 'Days between admission_date and discharge_date.';
ALTER TABLE claims.gold.claims_enriched ALTER COLUMN room_rent_over_cap COMMENT 'True if room_rent_per_day exceeded the plan''s room_rent_cap_per_day.';
ALTER TABLE claims.gold.claims_enriched ALTER COLUMN turnaround_days COMMENT 'Days from submitted_ts to settled_ts.';
ALTER TABLE claims.gold.claims_enriched ALTER COLUMN is_network_at_admission COMMENT 'True if the hospital was in the insurer network on admission_date.';
ALTER TABLE claims.gold.claims_enriched ALTER COLUMN hospital_city COMMENT 'City of the admitting hospital.';
ALTER TABLE claims.gold.claims_enriched ALTER COLUMN hospital_city_tier COMMENT 'City tier of the hospital: metro or non_metro.';

-- claims.gold.hospital_monthly 
COMMENT ON TABLE claims.gold.hospital_monthly IS
  'Monthly per-hospital claims rollup by admission month, aggregated from claims.gold.claims_enriched: claim and rejection counts plus billed and approved totals.';

ALTER TABLE claims.gold.hospital_monthly ALTER COLUMN hospital_id COMMENT 'Identifier of the hospital; joins to claims.raw.hospitals.';
ALTER TABLE claims.gold.hospital_monthly ALTER COLUMN admission_month COMMENT 'Month of admission (first day of the month); part of the table grain with hospital_id.';
ALTER TABLE claims.gold.hospital_monthly ALTER COLUMN claim_count COMMENT 'Number of claims admitted in the month.';
ALTER TABLE claims.gold.hospital_monthly ALTER COLUMN total_billed COMMENT 'Sum of billed_amount for claims admitted in the month.';
ALTER TABLE claims.gold.hospital_monthly ALTER COLUMN total_approved COMMENT 'Sum of approved_amount for claims admitted in the month.';
ALTER TABLE claims.gold.hospital_monthly ALTER COLUMN rejected_count COMMENT 'Number of claims rejected in the month.';

-- claims.gold.hospital_monthly_designer 
COMMENT ON TABLE claims.gold.hospital_monthly_designer IS
  'Monthly per-hospital claims rollup produced by the visual data prep (designer) flow';
ALTER TABLE claims.gold.hospital_monthly_designer ALTER COLUMN hospital_id COMMENT 'Identifier of the hospital; joins to claims.raw.hospitals.';
ALTER TABLE claims.gold.hospital_monthly_designer ALTER COLUMN admission_month COMMENT 'Month of admission (first day of the month); part of the table grain with hospital_id.';
ALTER TABLE claims.gold.hospital_monthly_designer ALTER COLUMN claim_count COMMENT 'Number of claims admitted in the month.';
ALTER TABLE claims.gold.hospital_monthly_designer ALTER COLUMN total_billed COMMENT 'Sum of billed_amount for claims admitted in the month.';
ALTER TABLE claims.gold.hospital_monthly_designer ALTER COLUMN total_approved COMMENT 'Sum of approved_amount for claims admitted in the month.';
ALTER TABLE claims.gold.hospital_monthly_designer ALTER COLUMN rejected_count COMMENT 'Number of claims rejected in the month.';

## Audit

In [0]:
create or replace table claims.gov.audit_claims_enriched as
select version, timestamp as changed_at, userName as changed_by, operation, "describe_history" as source
from (describe history claims.gold.claims_enriched)
order by version
limit 3